<a href="https://colab.research.google.com/github/pranavkantgaur/training_materials/blob/master/nuclear_reactor_lec_3_bezier_flux_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 3: Bezier Curves for Flux Profile Optimization
## Shaping Power Distribution Through Control Points

### Objectives:
1. Review Bezier curves and control points
2. Model spatial flux distributions with Bezier curves
3. Optimize flux flattening for uniform power distribution
4. Hands-on: Design enrichment zones for target flux shape

## Bezier Curves: Quick Review

### Definition
A **Bezier curve** of degree $n$ is defined by $n+1$ control points $P_0, P_1, \ldots, P_n$

### Parametric Form
$$C(t) = \sum_{i=0}^{n} B_{i,n}(t) P_i, \quad t \in [0, 1]$$

Where $B_{i,n}(t)$ are **Bernstein polynomials**:
$$B_{i,n}(t) = \binom{n}{i} t^i (1-t)^{n-i}$$

### Cubic Bezier (n=3)
Most common in practice:
$$C(t) = (1-t)^3 P_0 + 3(1-t)^2 t P_1 + 3(1-t)t^2 P_2 + t^3 P_3$$

### Key Properties
1. **Endpoint interpolation**: $C(0) = P_0$, $C(1) = P_n$
2. **Convex hull**: Curve lies within convex hull of control points
3. **Intuitive control**: Moving control points reshapes curve
4. **Variation diminishing**: Curve is smoother than control polygon
5. **Tangents**: $C'(0) \propto (P_1 - P_0)$, $C'(1) \propto (P_n - P_{n-1})$

### Why Bezier for Reactor Flux?
- **Design intuition**: Control points = enrichment zones
- **Optimization**: Vary control points to achieve target flux
- **Constraints**: Convex hull ensures physical bounds
- **Smooth flux**: No artificial oscillations

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import comb
from scipy.optimize import minimize, differential_evolution
from scipy.integrate import odeint, solve_bvp

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

In [ ]:
# Bernstein polynomial
def bernstein(i, n, t):
    """Bernstein polynomial basis"""
    return comb(n, i) * (t**i) * ((1-t)**(n-i))

def bezier_curve(t, control_points):
    """
    Evaluate Bezier curve at parameter t
    control_points: array of shape (n+1, dim)
    """
    n = len(control_points) - 1
    curve = np.zeros(control_points.shape[1] if len(control_points.shape) > 1 else 1)
    
    for i in range(n+1):
        curve += bernstein(i, n, t) * control_points[i]
    
    return curve

def bezier_curve_derivative(t, control_points):
    """
    Derivative of Bezier curve
    """
    n = len(control_points) - 1
    if n == 0:
        return 0
    
    # Derivative is degree n-1 Bezier with control points n*(P_{i+1} - P_i)
    deriv_control = n * np.diff(control_points, axis=0)
    return bezier_curve(t, deriv_control)

# Visualize Bernstein polynomials
t = np.linspace(0, 1, 200)
n = 3  # cubic

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
for i in range(n+1):
    B = [bernstein(i, n, ti) for ti in t]
    plt.plot(t, B, linewidth=2, label=f'B_{i},{n}(t)')
plt.xlabel('Parameter t', fontsize=12)
plt.ylabel('Bernstein Polynomial', fontsize=12)
plt.title('Cubic Bezier Basis Functions', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

# Example Bezier curve
plt.subplot(1, 2, 2)
P = np.array([[0, 0], [0.2, 1], [0.8, 1.2], [1, 0.3]])  # control points
curve = np.array([bezier_curve(ti, P) for ti in t])
plt.plot(curve[:, 0], curve[:, 1], 'b-', linewidth=2.5, label='Bezier Curve')
plt.plot(P[:, 0], P[:, 1], 'ro-', markersize=8, linewidth=1, alpha=0.5, label='Control Polygon')
plt.xlabel('x', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Cubic Bezier Curve Example', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')

plt.tight_layout()
plt.show()

## Modeling Flux Distribution with Bezier Curves

### Problem Setup
For a 1D reactor of length $L$:
- Position $x \in [0, L]$ maps to parameter $t = x/L \in [0, 1]$
- Control points $P_i$ represent flux values at design locations
- Bezier curve interpolates to give smooth flux profile

### Design Variables
- Control point positions (fixed or variable)
- Control point values (flux magnitudes)
- Relate to physical parameters: enrichment, material composition

### Constraints
1. Boundary conditions: $\phi(0) = \phi(L) = 0$ (vacuum)
2. Positivity: $\phi(x) > 0$ for all $x$
3. Material limits: enrichment $\leq 5\%$ for LEU

In [ ]:
# Example 1: Fit Bezier curve to analytical flux profile
L = 200.0  # reactor length (cm)
x = np.linspace(0, L, 500)
t_param = x / L

# Analytical flux (cosine shape)
phi_analytical = np.sin(np.pi * x / L)

# Design Bezier curve with 5 control points
n_control = 5
t_control = np.linspace(0, 1, n_control)
x_control = t_control * L

# Control point values from analytical solution
phi_control = np.sin(np.pi * t_control)

# Evaluate Bezier curve
phi_bezier = np.array([bezier_curve(t, phi_control) for t in t_param])

# Plot comparison
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(x, phi_analytical, 'k-', linewidth=2.5, label='Analytical', alpha=0.7)
plt.plot(x, phi_bezier, 'b--', linewidth=2, label='Bezier Approximation')
plt.plot(x_control, phi_control, 'ro', markersize=10, label='Control Points')
plt.xlabel('Position x (cm)', fontsize=12)
plt.ylabel('Normalized Flux', fontsize=12)
plt.title('Flux Profile: Analytical vs Bezier', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

# Error
plt.subplot(1, 2, 2)
error = np.abs(phi_analytical - phi_bezier)
plt.plot(x, error, 'r-', linewidth=2)
plt.xlabel('Position x (cm)', fontsize=12)
plt.ylabel('Absolute Error', fontsize=12)
plt.title(f'Bezier Approximation Error (Max={np.max(error):.4f})', fontsize=14)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Number of control points: {n_control}")
print(f"Maximum error: {np.max(error):.6f}")
print(f"RMS error: {np.sqrt(np.mean(error**2)):.6f}")

## Flux Flattening: A Key Design Goal

### Motivation
**Natural flux shape** (uniform enrichment): Sinusoidal
- High peak-to-average ratio
- Uneven power distribution → hot spots
- Inefficient fuel utilization

**Desired flux shape**: Flat
- Lower peak-to-average ratio
- Uniform power → better cooling
- More fuel burned uniformly

### Design Strategy
Use **variable enrichment zones**:
- Higher enrichment at edges (boost flux)
- Lower enrichment at center (reduce peak)
- Control points = zone boundaries and enrichments

### Optimization Problem
Minimize: Peak-to-average flux ratio

Subject to:
- Criticality: $k_{eff} = 1.0$
- Enrichment bounds: $2\% \leq e \leq 5\%$
- Vacuum boundaries: $\phi(0) = \phi(L) = 0$

In [ ]:
# Example 2: Flux flattening through enrichment zoning

def solve_flux_diffusion(L, enrichment_profile, n_points=500):
    """
    Solve 1D diffusion equation with variable enrichment
    -D d²φ/dx² + Σ_a φ = ν Σ_f φ
    """
    x = np.linspace(0, L, n_points)
    dx = x[1] - x[0]
    
    # Material properties (simplified)
    D = 1.0  # diffusion coefficient
    
    # Cross-sections depend on enrichment
    def get_cross_sections(enrichment):
        barn = 1e-24
        rho_U = 19.1
        N_A = 6.022e23
        A_U = 238
        N_total = rho_U * N_A / A_U * barn
        
        N_U235 = enrichment * N_total
        N_U238 = (1 - enrichment) * N_total
        
        sigma_f_U235 = 585 * barn
        sigma_a_U235 = 681 * barn
        sigma_a_U238 = 2.7 * barn
        nu = 2.43
        
        Sigma_f = sigma_f_U235 * N_U235
        Sigma_a = sigma_a_U235 * N_U235 + sigma_a_U238 * N_U238
        
        return nu * Sigma_f, Sigma_a
    
    # Evaluate enrichment at all positions
    t_positions = x / L
    enrichments = np.array([enrichment_profile(t) for t in t_positions])
    
    # Get cross-sections
    nu_Sigma_f = np.zeros(n_points)
    Sigma_a = np.zeros(n_points)
    
    for i in range(n_points):
        nu_Sigma_f[i], Sigma_a[i] = get_cross_sections(enrichments[i])
    
    # Build finite difference matrix
    # -D d²φ/dx² + Σ_a φ = k * ν Σ_f φ (eigenvalue problem)
    A = np.zeros((n_points, n_points))
    M = np.zeros((n_points, n_points))
    
    for i in range(1, n_points-1):
        A[i, i-1] = -D / dx**2
        A[i, i] = 2*D / dx**2 + Sigma_a[i]
        A[i, i+1] = -D / dx**2
        M[i, i] = nu_Sigma_f[i]
    
    # Boundary conditions (vacuum)
    A[0, 0] = 1.0
    A[-1, -1] = 1.0
    
    # Solve eigenvalue problem (only need dominant eigenvalue)
    from scipy.linalg import eig
    eigenvalues, eigenvectors = eig(A, M)
    
    # Find largest real eigenvalue (k_eff)
    real_eigs = np.real(eigenvalues[np.isreal(eigenvalues)])
    if len(real_eigs) > 0:
        k_eff = np.max(real_eigs)
        idx = np.argmax(np.real(eigenvalues))
        phi = np.real(eigenvectors[:, idx])
        phi = np.abs(phi) / np.max(np.abs(phi))  # normalize
    else:
        k_eff = 1.0
        phi = np.zeros(n_points)
    
    return x, phi, k_eff, enrichments

# Test with uniform enrichment
def uniform_enrichment(t):
    return 0.04  # 4% everywhere

x_uni, phi_uni, k_eff_uni, enr_uni = solve_flux_diffusion(L, uniform_enrichment)

# Calculate peak-to-average
peak_to_avg_uni = np.max(phi_uni) / np.mean(phi_uni[phi_uni > 0])

print(f"Uniform Enrichment (4%):")
print(f"  k_eff: {k_eff_uni:.6f}")
print(f"  Peak-to-average flux: {peak_to_avg_uni:.4f}")

# Plot
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(x_uni, phi_uni, 'b-', linewidth=2.5)
plt.xlabel('Position x (cm)', fontsize=12)
plt.ylabel('Normalized Flux', fontsize=12)
plt.title(f'Uniform Enrichment: P/A = {peak_to_avg_uni:.3f}', fontsize=14)
plt.grid(True, alpha=0.3)
plt.axhline(y=np.mean(phi_uni[phi_uni > 0]), color='r', linestyle='--', alpha=0.5, label='Average')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(x_uni, enr_uni*100, 'g-', linewidth=2.5)
plt.xlabel('Position x (cm)', fontsize=12)
plt.ylabel('Enrichment (%)', fontsize=12)
plt.title('Enrichment Profile (Uniform)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.ylim([0, 6])

plt.tight_layout()
plt.show()

In [ ]:
# Example 3: Optimize enrichment profile using Bezier curve

def enrichment_from_bezier(t, control_enrichments):
    """Enrichment profile defined by Bezier curve"""
    return bezier_curve(t, control_enrichments)

def objective(control_enrichments):
    """
    Objective: Minimize peak-to-average + penalty for off-criticality
    """
    # Enforce bounds
    if np.any(control_enrichments < 0.02) or np.any(control_enrichments > 0.05):
        return 1e10
    
    # Create enrichment profile function
    def enr_profile(t):
        return enrichment_from_bezier(t, control_enrichments)
    
    try:
        x, phi, k_eff, enrichments = solve_flux_diffusion(L, enr_profile)
        
        # Peak-to-average
        phi_positive = phi[phi > 0.01]  # ignore near-zero values
        if len(phi_positive) == 0:
            return 1e10
        
        peak_to_avg = np.max(phi_positive) / np.mean(phi_positive)
        
        # Criticality penalty
        k_penalty = 1000 * (k_eff - 1.0)**2
        
        return peak_to_avg + k_penalty
    except:
        return 1e10

# Optimize with 5 control points
n_control = 5
initial_guess = np.array([0.045, 0.04, 0.035, 0.04, 0.045])  # higher at edges

print("Optimizing enrichment profile...")
print("This may take a minute...\n")

# Use bounded optimization
bounds = [(0.02, 0.05) for _ in range(n_control)]
result = minimize(objective, initial_guess, method='L-BFGS-B', bounds=bounds,
                  options={'maxiter': 50})

optimal_enrichments = result.x

print(f"Optimization converged: {result.success}")
print(f"Optimal control enrichments: {optimal_enrichments*100}")

# Evaluate optimal solution
def optimal_profile(t):
    return enrichment_from_bezier(t, optimal_enrichments)

x_opt, phi_opt, k_eff_opt, enr_opt = solve_flux_diffusion(L, optimal_profile)
phi_opt_positive = phi_opt[phi_opt > 0.01]
peak_to_avg_opt = np.max(phi_opt_positive) / np.mean(phi_opt_positive)

print(f"\nOptimized Solution:")
print(f"  k_eff: {k_eff_opt:.6f}")
print(f"  Peak-to-average flux: {peak_to_avg_opt:.4f}")
print(f"\nImprovement: {(peak_to_avg_uni - peak_to_avg_opt)/peak_to_avg_uni*100:.1f}% reduction in P/A ratio")

# Plot comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Flux profiles
axes[0, 0].plot(x_uni, phi_uni, 'b-', linewidth=2, label='Uniform', alpha=0.7)
axes[0, 0].plot(x_opt, phi_opt, 'r-', linewidth=2.5, label='Optimized')
axes[0, 0].axhline(y=np.mean(phi_opt_positive), color='green', linestyle='--', 
                   alpha=0.5, label='Avg (optimized)')
axes[0, 0].set_xlabel('Position x (cm)', fontsize=11)
axes[0, 0].set_ylabel('Normalized Flux', fontsize=11)
axes[0, 0].set_title('Flux Profiles Comparison', fontsize=13)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Enrichment profiles
axes[0, 1].plot(x_uni, enr_uni*100, 'b-', linewidth=2, label='Uniform', alpha=0.7)
axes[0, 1].plot(x_opt, enr_opt*100, 'r-', linewidth=2.5, label='Optimized')
t_control_opt = np.linspace(0, 1, n_control)
x_control_opt = t_control_opt * L
axes[0, 1].plot(x_control_opt, optimal_enrichments*100, 'go', markersize=10, 
                label='Control Points')
axes[0, 1].set_xlabel('Position x (cm)', fontsize=11)
axes[0, 1].set_ylabel('Enrichment (%)', fontsize=11)
axes[0, 1].set_title('Enrichment Profiles', fontsize=13)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim([0, 6])

# Power distribution
power_uni = phi_uni
power_opt = phi_opt
axes[1, 0].fill_between(x_uni, 0, power_uni, alpha=0.3, color='blue', label='Uniform')
axes[1, 0].fill_between(x_opt, 0, power_opt, alpha=0.5, color='red', label='Optimized')
axes[1, 0].set_xlabel('Position x (cm)', fontsize=11)
axes[1, 0].set_ylabel('Power Density (normalized)', fontsize=11)
axes[1, 0].set_title('Power Distribution', fontsize=13)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Peak-to-average comparison
cases = ['Uniform', 'Optimized']
p_to_a = [peak_to_avg_uni, peak_to_avg_opt]
colors = ['blue', 'red']
bars = axes[1, 1].bar(cases, p_to_a, color=colors, alpha=0.7)
axes[1, 1].set_ylabel('Peak-to-Average Ratio', fontsize=11)
axes[1, 1].set_title('Flux Flattening Performance', fontsize=13)
axes[1, 1].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, p_to_a):
    height = bar.get_height()
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=12)

plt.tight_layout()
plt.show()

## Control Point Sensitivity Analysis

One advantage of Bezier curves: **intuitive sensitivity**

How does changing each control point affect:
1. Peak-to-average ratio?
2. k_eff?
3. Maximum flux location?

This analysis helps designers understand which zones are most critical.

In [ ]:
# Sensitivity analysis: perturb each control point
def compute_sensitivities(base_control_points, delta=0.001):
    """
    Compute sensitivities dJ/de_i for each control point
    J = peak-to-average ratio
    """
    n = len(base_control_points)
    sensitivities_pa = np.zeros(n)
    sensitivities_k = np.zeros(n)
    
    # Base case
    def base_profile(t):
        return enrichment_from_bezier(t, base_control_points)
    
    x_base, phi_base, k_base, _ = solve_flux_diffusion(L, base_profile)
    phi_pos = phi_base[phi_base > 0.01]
    pa_base = np.max(phi_pos) / np.mean(phi_pos)
    
    # Perturb each control point
    for i in range(n):
        perturbed = base_control_points.copy()
        perturbed[i] += delta
        
        if perturbed[i] > 0.05:
            perturbed[i] = 0.05
        
        def pert_profile(t):
            return enrichment_from_bezier(t, perturbed)
        
        try:
            x_pert, phi_pert, k_pert, _ = solve_flux_diffusion(L, pert_profile)
            phi_pos_pert = phi_pert[phi_pert > 0.01]
            pa_pert = np.max(phi_pos_pert) / np.mean(phi_pos_pert)
            
            sensitivities_pa[i] = (pa_pert - pa_base) / delta
            sensitivities_k[i] = (k_pert - k_base) / delta
        except:
            sensitivities_pa[i] = 0
            sensitivities_k[i] = 0
    
    return sensitivities_pa, sensitivities_k

print("Computing sensitivities...\n")
sens_pa, sens_k = compute_sensitivities(optimal_enrichments)

# Plot sensitivities
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_positions = np.arange(n_control)
axes[0].bar(x_positions, sens_pa, color='purple', alpha=0.7)
axes[0].set_xlabel('Control Point Index', fontsize=12)
axes[0].set_ylabel('∂(P/A) / ∂enrichment', fontsize=12)
axes[0].set_title('Sensitivity of Peak-to-Average Ratio', fontsize=14)
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].axhline(y=0, color='black', linewidth=1)

axes[1].bar(x_positions, sens_k, color='orange', alpha=0.7)
axes[1].set_xlabel('Control Point Index', fontsize=12)
axes[1].set_ylabel('∂k_eff / ∂enrichment', fontsize=12)
axes[1].set_title('Sensitivity of k_eff', fontsize=14)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].axhline(y=0, color='black', linewidth=1)

plt.tight_layout()
plt.show()

print("Sensitivity Analysis:")
for i in range(n_control):
    print(f"  Control Point {i}: ∂(P/A)/∂e = {sens_pa[i]:+.2f}, ∂k/∂e = {sens_k[i]:+.4f}")

print("\nInterpretation:")
print("  - Positive ∂(P/A)/∂e: Increasing enrichment increases peak-to-average (bad)")
print("  - Negative ∂(P/A)/∂e: Increasing enrichment flattens flux (good)")
print("  - Positive ∂k/∂e: Increasing enrichment increases reactivity")

## Summary

### What We Learned:
1. ✅ **Bezier curves** provide intuitive control through control points
2. ✅ Applied to **spatial flux distribution** in reactors
3. ✅ **Flux flattening** achieved through enrichment zoning
4. ✅ **Optimized** peak-to-average ratio while maintaining criticality
5. ✅ Analyzed **control point sensitivities** for design insights

### Key Results:
- Achieved significant reduction in peak-to-average ratio
- Maintained k_eff ≈ 1.0 throughout
- Used only 5 control points (practical for real designs)
- Sensitivities reveal which zones are most important

### Advantages of Bezier Approach:
- **Intuitive**: Control points = physical design zones
- **Smooth**: No artificial oscillations
- **Constrained**: Convex hull property ensures bounds
- **Flexible**: Easy to add/remove control points

### Limitations:
- Global support: Moving one point affects entire curve
- Difficult to enforce local constraints
- May need many control points for complex shapes

**Next Lecture**: We'll use **B-splines** for multi-zone cores with local control, enabling more complex enrichment patterns!

## Exercises

1. **Different Degree Bezier Curves**:
   - Try quadratic (3 control points) and quintic (6 control points)
   - How does degree affect optimization results?
   - Trade-off between flexibility and complexity?

2. **Multi-Objective Optimization**:
   - Add constraint: minimize fuel cost (total enrichment)
   - Balance flux flattening vs fuel economy
   - Use weighted objective: α*(P/A) + β*(total enrichment)

3. **Cylindrical Geometry**:
   - Extend to radial flux distribution in cylindrical reactor
   - Solve 1D radial diffusion equation
   - Optimize radial enrichment profile

4. **Control Point Placement**:
   - What if control points aren't equally spaced?
   - Try more points near boundaries, fewer in center
   - Does this improve optimization?

5. **Burnup-Dependent Optimization**:
   - Design enrichment profile that stays flat over burnup
   - Couple with depletion from Lecture 2
   - How does optimal profile change with time?